# Speculative decoding: draft, verify và đo cho trung thực

Decode bị chặn bởi băng thông bộ nhớ (module 05 mục 5, module 21 mục 4): mỗi bước đọc toàn bộ trọng số để sinh **một**
token. Speculative decoding đoán trước vài token bằng một cách rẻ, rồi cho model lớn **kiểm cả khối trong một lần
forward**. Với greedy, kết quả phải giống hệt từng token so với chạy model lớn một mình; cái thay đổi chỉ là số lần gọi
model lớn. Bài này cài thuật toán, hai cách đoán (prompt lookup và model nháp), và đo khi nào nó nhanh hơn, khi nào chậm hơn.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) nạp model Hugging Face và đo; test bỏ qua chúng.
- Cần `Qwen/Qwen2.5-0.5B-Instruct` (khoảng 1 GB, tự tải về cache Hugging Face ở lần chạy đầu) và bộ `zalo-legal`.
  Phần đo chạy trên CPU 4 luồng, khoảng 4 phút trên PC của track. Phần mở rộng 3B cần khoảng 14 GB RAM, tắt mặc định.
- Chấm: `uv run pytest tracks/ai-engineer/modules/20-evaluation-production/exercises/intermediate/03-speculative-decoding -q`

In [ ]:
# explore: đưa gốc repo vào sys.path, tìm thư mục dữ liệu
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
ZALO = ROOT / "data" / "raw" / "zalo-legal"

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import numpy as np

## 1. Giao diện model và greedy

Mọi model trong bài (model Hugging Face thật, model giả trong test) có cùng giao diện **có trạng thái**, giống một KV cache:

| Phương thức | Ý nghĩa |
| --- | --- |
| `lm.reset(prompt_ids) -> logits` | xoá trạng thái, xử lý cả prompt (prefill), trả logits ở vị trí cuối, shape `(vocab,)` |
| `lm.extend(ids) -> logits` | nối `ids` vào trạng thái trong **một** lần forward, trả logits ở **mỗi** vị trí vừa nối, shape `(len(ids), vocab)`; dòng `i` dự đoán token đứng sau `ids[i]` |
| `lm.truncate(length)` | giữ `length` token đầu của trạng thái, bỏ phần sau (cắt KV cache) |

Mỗi lần gọi `reset` hoặc `extend` là một lần forward của model: đó là đơn vị chi phí mà bài này đếm.

`greedy_generate(lm, prompt_ids, max_new_tokens, eos_id=None) -> list[int]`: token đầu là argmax của logits do `reset`
trả; mỗi token sau là argmax của logits do `extend([token vừa sinh])` trả. Dừng khi vừa sinh `eos_id` (giữ `eos_id` trong
kết quả) hoặc đủ `max_new_tokens` token. Không gọi `extend` thừa sau token cuối.

In [ ]:
def greedy_generate(lm, prompt_ids: list[int], max_new_tokens: int, eos_id: int | None = None) -> list[int]:
    """Greedy với KV cache: một reset rồi mỗi token một extend."""
    # TODO
    raise NotImplementedError

## 2. Prompt lookup: đoán bằng chính ngữ cảnh

Khi câu trả lời chép lại một phần prompt (trích nguyên văn điều luật, sửa code, tóm tắt có trích dẫn), token sắp tới rất
có thể là token đã đứng sau cùng một cụm ở đâu đó trong ngữ cảnh. Prompt lookup (Saxena, 2023; có sẵn trong Transformers
và llama.cpp) không cần model nháp nào.

`prompt_lookup_draft(context, k, max_ngram=3) -> list[int]`: với `n` từ `min(max_ngram, len(context) - 1)` giảm dần về 1,
lấy `n` token cuối của `context` làm mẫu và tìm lần xuất hiện **gần cuối nhất** của mẫu đó, bắt đầu ở vị trí
`start <= len(context) - n - 1` (không tính chính đuôi). Thấy thì trả `context[start + n : start + n + k]` (có thể ngắn
hơn `k`); không thấy với mọi `n` thì `[]`. `k <= 0` trả `[]`.

In [ ]:
def prompt_lookup_draft(context: list[int], k: int, max_ngram: int = 3) -> list[int]:
    """Tối đa k token đứng sau lần xuất hiện gần cuối nhất của n-gram đuôi (n giảm dần)."""
    # TODO
    raise NotImplementedError

## 3. Draft rồi verify

`speculative_generate(target, draft, prompt_ids, k, max_new_tokens, eos_id=None) -> (tokens, stats)`. `draft(context, m)`
là hàm đoán, trả list tối đa `m` token đi tiếp sau `context` (có thể rỗng). Thuật toán greedy:

1. `target.reset(prompt)`; token kế `t` là argmax. `t` luôn đúng (đó là lựa chọn của target).
2. Lặp: nối `t` vào output; dừng nếu `t == eos_id` hoặc đủ `max_new_tokens`. Gọi `draft(prompt + output, m)` với
   `m = min(k, max_new_tokens - len(output))`, chỉ giữ tối đa `m` token đầu nó trả. Gọi **một** lần
   `target.extend([t] + đoán)`. Duyệt các token đoán theo thứ tự: token đoán thứ `i` được nhận nếu nó bằng argmax của dòng
   `i` của logits vừa trả; nhận thì nối vào output (dừng ngay nếu nó là `eos_id` hoặc output đã đủ), sai thì dừng duyệt.
   Gọi `a` là số token đoán được nhận: token kế `t` là argmax của dòng `a` (token sửa khi đoán sai, hoặc token "thưởng"
   khi đoán đúng hết). Cuối vòng, cắt trạng thái target về `len(prompt) + len(output)` để bỏ phần đoán sai.

`stats` là dict: `target_calls` (số lần gọi `reset` + `extend` của target), `rounds` (số lần `extend` có ít nhất một token
đoán), `proposed` (tổng số token đoán đã gửi kiểm), `accepted` (tổng số token đoán được nhận), `acceptance_rate`
(`accepted / proposed`, bằng `0.0` khi chưa đoán gì).

Kết quả phải **trùng từng token** với `greedy_generate(target, ...)` dù hàm đoán tốt hay tệ: chỉ tốc độ thay đổi.

In [ ]:
def speculative_generate(target, draft, prompt_ids: list[int], k: int, max_new_tokens: int,
                         eos_id: int | None = None) -> tuple[list[int], dict]:
    """Greedy speculative decoding: kết quả trùng greedy của target, kèm số liệu đoán/nhận."""
    # TODO
    raise NotImplementedError

## 4. Đo trên Qwen2.5-0.5B với điều luật Zalo

Harness dưới đây (không phải bài tập) bọc model Hugging Face theo giao diện ở mục 1 bằng `DynamicCache`. Ở Transformers
5.22, `cache.crop(n)` nhận số **âm** là số token cần bỏ (số dương báo lỗi), nên `truncate` tính phần chênh lệch.
`ModelDrafter` biến một model nhỏ thành hàm đoán: giữ KV cache riêng, đồng bộ với ngữ cảnh mỗi lần được gọi, rồi sinh
greedy `m` token.

Hai loại tác vụ trên cùng một điều luật (4 điều đầu của corpus có độ dài 1 200 tới 1 600 ký tự và có khoản 2):
**copy** (chép nguyên văn khoản 2) và **free** (giải thích bằng lời của mình). Mỗi cấu hình sinh tối đa 128 token, CPU,
float32, 4 luồng.

In [ ]:
# explore: harness Hugging Face theo giao diện reset / extend / truncate
import json
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

torch.set_num_threads(4)


class HFLM:
    def __init__(self, model):
        self.model, self.cache, self.n = model, None, 0

    @torch.no_grad()
    def _forward(self, ids):
        out = self.model(input_ids=torch.tensor([ids]), past_key_values=self.cache, use_cache=True)
        self.n += len(ids)
        return out.logits[0].float().numpy()

    def reset(self, ids):
        self.cache, self.n = DynamicCache(), 0
        return self._forward(list(ids))[-1]

    def extend(self, ids):
        return self._forward(list(ids))

    def truncate(self, length):
        if self.n > length:
            self.cache.crop(length - self.n)
        self.n = length


class ModelDrafter:
    def __init__(self, lm):
        self.lm, self.tokens = lm, []

    def __call__(self, context, m):
        context = list(context)
        common = 0
        for a, b in zip(self.tokens, context):
            if a != b:
                break
            common += 1
        common = min(common, len(context) - 1)
        if common == 0:
            logits = self.lm.reset(context)
        else:
            self.lm.truncate(common)
            logits = self.lm.extend(context[common:])[-1]
        self.tokens = list(context)
        out = []
        for i in range(m):
            out.append(int(np.argmax(logits)))
            if i < m - 1:
                logits = self.lm.extend([out[-1]])[-1]
                self.tokens.append(out[-1])
        return out


def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    return tok, HFLM(AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval())


def make_tasks(tok, n_docs):
    docs = []
    with (ZALO / "corpus.jsonl").open(encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            if 1200 <= len(d["text"]) <= 1600 and "\n2." in d["text"]:
                docs.append(d)
            if len(docs) == n_docs:
                break
    tasks = []
    for d in docs:
        article = d["title"] + "\n" + d["text"]
        for kind, instruction in [
            ("copy", "Chép lại nguyên văn khoản 2 của điều luật trên, không thêm bớt chữ nào."),
            ("free", "Giải thích điều luật trên cho một người không học luật, bằng lời của bạn, có ví dụ."),
        ]:
            ids = tok.apply_chat_template([{"role": "user", "content": f"{article}\n\n{instruction}"}],
                                          add_generation_prompt=True)
            ids = ids if isinstance(ids, list) else ids["input_ids"]
            tasks.append((d["_id"], kind, list(ids)))
    return tasks

In [ ]:
# explore: greedy so với prompt lookup (k = 4 và 8) trên 8 tác vụ, khoảng 4 phút trên CPU
tok, target = load("Qwen/Qwen2.5-0.5B-Instruct")
EOS = tok.convert_tokens_to_ids("<|im_end|>")
tasks = make_tasks(tok, 4)
greedy_generate(target, tasks[0][2][:50], 4)

rows = []
for doc, kind, ids in tasks:
    t0 = time.perf_counter()
    ref = greedy_generate(target, ids, 128, EOS)
    greedy_s = time.perf_counter() - t0
    for k in (4, 8):
        t0 = time.perf_counter()
        out, st = speculative_generate(target, prompt_lookup_draft, ids, k, 128, EOS)
        spec_s = time.perf_counter() - t0
        rows.append(dict(doc=doc, kind=kind, k=k, prompt=len(ids), n_out=len(ref), same=out == ref,
                         greedy_s=round(greedy_s, 2), spec_s=round(spec_s, 2), speedup=round(greedy_s / spec_s, 2),
                         **{key: round(v, 3) for key, v in st.items()}))
        print(rows[-1])

import pandas as pd

df = pd.DataFrame(rows)
print(df.groupby(["kind", "k"])[["speedup", "acceptance_rate", "target_calls"]].agg(["min", "median", "max"]))
print("trùng greedy:", int(df.same.sum()), "/", len(df))

In [ ]:
# explore: mở rộng (không chấm): Qwen2.5-0.5B đoán cho Qwen2.5-3B, k = 4, 64 token, CPU float32.
# Cần khoảng 14 GB RAM và 5 phút. Bật bằng biến môi trường M20_DRAFT_3B=1 trước khi mở notebook.
if os.environ.get("M20_DRAFT_3B") == "1":
    tok3, big = load("Qwen/Qwen2.5-3B-Instruct")
    drafter = ModelDrafter(load("Qwen/Qwen2.5-0.5B-Instruct")[1])
    greedy_generate(big, tasks[0][2][:50], 4)
    for doc, kind, ids in make_tasks(tok3, 2):
        t0 = time.perf_counter()
        ref = greedy_generate(big, ids, 64, EOS)
        greedy_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        out, st = speculative_generate(big, drafter, ids, 4, 64, EOS)
        spec_s = time.perf_counter() - t0
        print(kind, "trùng:", out == ref, "tăng tốc:", round(greedy_s / spec_s, 2), st)

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Prompt lookup tăng tốc bao nhiêu ở tác vụ copy, bao nhiêu ở tác vụ free? Tỉ lệ nhận khác nhau thế nào?
- Tăng `k` từ 4 lên 8 đổi tỉ lệ nhận và tốc độ thế nào ở từng loại tác vụ? Vì sao tỉ lệ nhận và tốc độ không đi cùng chiều?
- Ở tác vụ free tỉ lệ nhận thấp mà vẫn không chậm hơn greedy: đo thời gian `target.extend` với 1 token và với 9 token để
  giải thích (công thức tăng tốc ở mục 5 lý thuyết).
- Output có trùng greedy ở mọi tác vụ không? Nếu chạy float16 trên GPU, điều đó còn chắc chắn không?
- (Mở rộng) Model nháp 0.5B cho target 3B trên CPU: vì sao tỉ lệ nhận cao ở copy mà tăng tốc vẫn nhỏ?

Sau đó làm phần trắc nghiệm của bài trong app.